In [1]:
# This notebook runs the complete NewsLens pipeline from
# start to finish in one place. It covers all four components:
# topic classification, churn prediction, article recommendation
# and the re-engagement pipeline that combines all three.
# This is the notebook to show in a portfolio or interview.

print("="*60)
print("  NewsLens — Publishing Intelligence Pipeline")
print("  Built to mirror Upscore's product suite")
print("="*60)

  NewsLens — Publishing Intelligence Pipeline
  Built to mirror Upscore's product suite


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import pickle
import re
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")
print("All libraries imported!")

All libraries imported!


In [3]:
# All settings in one place so they are easy to change.
# This is good practice for any production pipeline.

CONFIG = {
    "news_data_path"    : "../data/processed/news_clean.csv",
    "readers_path"      : "../data/processed/readers.csv",
    "sample_size"       : 50000,   # articles for topic model
    "rec_sample_size"   : 5000,    # articles for recommender
    "chunk_per_category": 12500,   # balanced sampling
    "churn_readers"     : 5000,    # simulated readers
    "tfidf_features"    : 10000,   # vocabulary size
    "rec_features"      : 5000,    # recommender vocabulary
    "test_size"         : 0.2,
    "random_state"      : 42,
    "churn_rate"        : 0.30,    # target churn rate
}

print("Configuration loaded!")
for k, v in CONFIG.items():
    print(f"  {k:25s}: {v}")

Configuration loaded!
  news_data_path           : ../data/processed/news_clean.csv
  readers_path             : ../data/processed/readers.csv
  sample_size              : 50000
  rec_sample_size          : 5000
  chunk_per_category       : 12500
  churn_readers            : 5000
  tfidf_features           : 10000
  rec_features             : 5000
  test_size                : 0.2
  random_state             : 42
  churn_rate               : 0.3


In [4]:
# Load the cleaned news data and prepare it for modelling.
# We do balanced sampling so no category dominates the training.

print("\nSTEP 1: Loading data...")

df_news = pd.read_csv(CONFIG["news_data_path"])
print(f"  Loaded {len(df_news):,} articles")

def clean_title(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_news["title_clean"] = df_news["title"].apply(clean_title)

# Balanced sample across categories
df_sample = df_news.groupby("category").apply(
    lambda x: x.sample(
        min(len(x), CONFIG["chunk_per_category"]),
        random_state=CONFIG["random_state"]
    )
).reset_index(drop=True)

print(f"  Sampled {len(df_sample):,} articles (balanced)")
print()
print("  Category breakdown:")
print(df_sample["category_name"].value_counts().to_string())


STEP 1: Loading data...
  Loaded 422,419 articles
  Sampled 50,000 articles (balanced)

  Category breakdown:
category_name
Business          12500
Entertainment     12500
Health            12500
Science & Tech    12500


/var/folders/d0/ymvkftpj5hzdbp0bfg1vxwvr0000gn/T/ipykernel_80320/768661714.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df_news.groupby("category").apply(


In [5]:
# Train the topic classifier and track with MLflow.
# We use TF-IDF features and Logistic Regression
# which gives the best accuracy for this task.

print("\nSTEP 2: Topic classification...")

mlflow.set_experiment("newslens-full-pipeline")

X_text = df_sample["title_clean"]
y_cat = df_sample["category"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_text, y_cat,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=y_cat
)

topic_tfidf = TfidfVectorizer(
    max_features=CONFIG["tfidf_features"],
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2
)

X_train_tfidf = topic_tfidf.fit_transform(X_train_t)
X_test_tfidf = topic_tfidf.transform(X_test_t)

with mlflow.start_run(run_name="topic-classifier"):

    topic_model = LogisticRegression(
        C=1.0, max_iter=1000, multi_class="multinomial"
    )
    topic_model.fit(X_train_tfidf, y_train_t)

    topic_accuracy = accuracy_score(
        y_test_t, topic_model.predict(X_test_tfidf)
    )
    mlflow.log_metric("topic_accuracy", topic_accuracy)
    mlflow.sklearn.log_model(topic_model, "topic_model")

print(f"  Topic classifier accuracy: {topic_accuracy:.4f}")

2026/10/23 13:20:22 INFO mlflow.tracking.fluent: Experiment with name 'newslens-full-pipeline' does not exist. Creating a new experiment.



STEP 2: Topic classification...


/Users/haritharetnakaran/Documents/Personal_projects/newslens-publishing-intelligence/venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2026/10/23 13:20:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/23 13:20:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  Topic classifier accuracy: 0.9068


In [6]:
# Simulate reader behaviour data and train the churn model.
# We track this run separately in MLflow so we can
# compare churn model versions independently.

print("\nSTEP 3: Churn prediction...")

np.random.seed(CONFIG["random_state"])
n = CONFIG["churn_readers"]

articles_per_week    = np.random.poisson(lam=7, size=n)
avg_session_minutes  = np.random.exponential(scale=8, size=n)
days_since_last      = np.random.exponential(scale=5, size=n)
weeks_subscribed     = np.random.randint(1, 104, size=n)
topic_diversity      = np.random.randint(1, 5, size=n)
newsletter_opens     = np.random.poisson(lam=3, size=n)
uses_app             = np.random.binomial(1, 0.4, size=n)

churn_score = (
    - 0.3 * articles_per_week
    - 0.2 * avg_session_minutes
    + 0.4 * days_since_last
    - 0.1 * weeks_subscribed
    - 0.1 * newsletter_opens
    + np.random.normal(0, 1, size=n)
)

churn = (churn_score > np.percentile(churn_score, 70)).astype(int)

df_readers = pd.DataFrame({
    "reader_id"           : [f"reader_{i}" for i in range(n)],
    "articles_per_week"   : articles_per_week,
    "avg_session_minutes" : avg_session_minutes.round(1),
    "days_since_last_visit": days_since_last.round(1),
    "weeks_subscribed"    : weeks_subscribed,
    "topic_diversity"     : topic_diversity,
    "newsletter_opens"    : newsletter_opens,
    "uses_app"            : uses_app,
    "churn"               : churn
})

X_churn = df_readers.drop(columns=["reader_id", "churn"])
y_churn = df_readers["churn"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_churn, y_churn,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=y_churn
)

scaler = StandardScaler()
X_train_cs = scaler.fit_transform(X_train_c)
X_test_cs  = scaler.transform(X_test_c)

with mlflow.start_run(run_name="churn-model"):

    churn_model = GradientBoostingClassifier(
        n_estimators=100, max_depth=4,
        random_state=CONFIG["random_state"]
    )
    churn_model.fit(X_train_cs, y_train_c)

    churn_auc = roc_auc_score(
        y_test_c,
        churn_model.predict_proba(X_test_cs)[:, 1]
    )
    mlflow.log_metric("churn_roc_auc", churn_auc)
    mlflow.sklearn.log_model(churn_model, "churn_model")

print(f"  Churn model ROC AUC: {churn_auc:.4f}")
print(f"  Churn rate         : {churn.mean():.1%}")


STEP 3: Churn prediction...


2026/10/23 13:20:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/23 13:20:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  Churn model ROC AUC: 0.9769
  Churn rate         : 30.0%


In [7]:
# Build the article recommendation engine using
# TF-IDF and cosine similarity on a sample of articles.

print("\nSTEP 4: Building recommender...")

df_rec = df_news.sample(
    CONFIG["rec_sample_size"],
    random_state=CONFIG["random_state"]
).reset_index(drop=True)

df_rec["title_clean"] = df_rec["title"].apply(clean_title)

rec_tfidf = TfidfVectorizer(
    max_features=CONFIG["rec_features"],
    ngram_range=(1, 2),
    stop_words="english"
)

rec_matrix = rec_tfidf.fit_transform(df_rec["title_clean"])
sim_matrix = cosine_similarity(rec_matrix, rec_matrix)

print(f"  Recommender built on {len(df_rec):,} articles")
print(f"  Similarity matrix   : {sim_matrix.shape}")


STEP 4: Building recommender...
  Recommender built on 5,000 articles
  Similarity matrix   : (5000, 5000)


In [8]:
# The re-engagement pipeline combines all three components.
# For any reader we predict their churn risk, classify
# what topics they might enjoy, and recommend articles.

category_map = {
    "b": "Business",
    "t": "Science & Tech",
    "e": "Entertainment",
    "m": "Health"
}

def get_category_recommendations(category_code, n=3):
    cat_df = df_rec[df_rec["category"] == category_code].copy()
    if len(cat_df) == 0:
        return []
    indices = cat_df.index.tolist()
    cat_sim = sim_matrix[np.ix_(indices, indices)]
    avg_sim = cat_sim.mean(axis=1)
    cat_df["score"] = avg_sim
    top = cat_df.nlargest(n, "score")
    return top["title"].tolist()


def full_pipeline(reader_id):
    reader = df_readers[df_readers["reader_id"] == reader_id].iloc[0]

    features = reader[[
        "articles_per_week", "avg_session_minutes",
        "days_since_last_visit", "weeks_subscribed",
        "topic_diversity", "newsletter_opens", "uses_app"
    ]].values.reshape(1, -1)

    churn_prob = churn_model.predict_proba(
        scaler.transform(features)
    )[0][1]

    is_at_risk = churn_prob > 0.5

    print(f"Reader         : {reader_id}")
    print(f"Churn risk     : {churn_prob:.1%}")
    print(f"Status         : {'⚠️  AT RISK' if is_at_risk else '✅ ENGAGED'}")
    print()

    if is_at_risk:
        print("Recommended articles to re-engage:")
        print()
        for code in ["b", "t", "e", "m"]:
            recs = get_category_recommendations(code, n=2)
            if recs:
                print(f"  {category_map[code]}:")
                for r in recs:
                    print(f"    • {r}")
        print()


print("STEP 5: Running full pipeline for sample readers")
print()

for reader_id in ["reader_0", "reader_50", "reader_200"]:
    print("="*60)
    full_pipeline(reader_id)

STEP 5: Running full pipeline for sample readers

Reader         : reader_0
Churn risk     : 94.1%
Status         : ⚠️  AT RISK

Recommended articles to re-engage:

  Business:
    • Stocks Hold Gains As Oil Stocks Rise, Airlines Recoup
    • Stocks move mostly higher on Wall Street
  Science & Tech:
    • Apple strikes a new chord
    • REVIEW: Samsung Galaxy S5
  Entertainment:
    • Video Game Review: Kim Kardashian: Hollywood
    • New trailer for Game Of Thrones Season 4: watch now
  Health:
    • Ebola outbreak feared to worsen
    • Ebola Outbreak: Australia and New Zealand Brace to Tackle Virus

Reader         : reader_50
Churn risk     : 0.1%
Status         : ✅ ENGAGED

Reader         : reader_200
Churn risk     : 0.5%
Status         : ✅ ENGAGED



/Users/haritharetnakaran/Documents/Personal_projects/newslens-publishing-intelligence/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/haritharetnakaran/Documents/Personal_projects/newslens-publishing-intelligence/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/haritharetnakaran/Documents/Personal_projects/newslens-publishing-intelligence/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [9]:
# Save all models and data so the Streamlit app
# can load them without rerunning the pipeline.

print("STEP 6: Saving all artifacts...")

os.makedirs("../data/processed", exist_ok=True)

with open("../data/processed/topic_tfidf.pkl", "wb") as f:
    pickle.dump(topic_tfidf, f)

with open("../data/processed/topic_model.pkl", "wb") as f:
    pickle.dump(topic_model, f)

with open("../data/processed/churn_model.pkl", "wb") as f:
    pickle.dump(churn_model, f)

with open("../data/processed/churn_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("../data/processed/rec_tfidf.pkl", "wb") as f:
    pickle.dump(rec_tfidf, f)

with open("../data/processed/sim_matrix.pkl", "wb") as f:
    pickle.dump(sim_matrix, f)

df_readers.to_csv("../data/processed/readers.csv", index=False)
df_rec.to_csv("../data/processed/rec_articles.csv", index=False)

print("  All artifacts saved to data/processed/")

STEP 6: Saving all artifacts...
  All artifacts saved to data/processed/


In [10]:
# All three components are now working together as one pipeline.
# The next step is converting everything into Python modules
# and building the Streamlit dashboard for editorial teams.

print()
print("="*60)
print("  NewsLens Pipeline — Complete!")
print("="*60)
print()
print(f"  Topic classifier accuracy : {topic_accuracy:.4f}")
print(f"  Churn model ROC AUC       : {churn_auc:.4f}")
print(f"  Recommender articles      : {len(df_rec):,}")
print()
print("  All models saved and ready for the Streamlit app!")
print()
print("  Next: src/ modules and Streamlit dashboard")


  NewsLens Pipeline — Complete!

  Topic classifier accuracy : 0.9068
  Churn model ROC AUC       : 0.9769
  Recommender articles      : 5,000

  All models saved and ready for the Streamlit app!

  Next: src/ modules and Streamlit dashboard
